# 05. XGBoost Regressor

In [ ]:
import sys
from pathlib import Path
import optuna
import pandas as pd
from sklearn.metrics import mean_absolute_error

sys.path.insert(0, str(Path.cwd().parent))
from src.models import get_xgboost, save_model
from src.evaluation import regression_metrics
from src.utils import set_seed, save_json

set_seed(42)
FEAT_DIR = Path('../data_features')
MODEL_DIR = Path('../models'); MODEL_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR = Path('../results/metrics'); RES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train = pd.read_parquet(FEAT_DIR / 'train.parquet')
val = pd.read_parquet(FEAT_DIR / 'val.parquet')
test = pd.read_parquet(FEAT_DIR / 'test.parquet')

TARGET = 'delay_hours'
DROP = [TARGET, 'trip_id', 'load_id', 'driver_id', 'truck_id', 'trailer_id',
        'customer_id', 'route_id', 'dispatch_date']
feature_cols = [c for c in train.columns if c not in DROP
                 and pd.api.types.is_numeric_dtype(train[c])]

X_train, y_train = train[feature_cols], train[TARGET]
X_val, y_val = val[feature_cols], val[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]

## 1. Tune với Optuna

In [ ]:
def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 200, 1000, step=100),
        max_depth=trial.suggest_int('max_depth', 3, 10),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 5.0, log=True),
    )
    m = get_xgboost(**params, early_stopping_rounds=30)
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return mean_absolute_error(y_val, m.predict(X_val))

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)
print('Best:', study.best_params)

## 2. Train final + save

In [ ]:
best_model = get_xgboost(**study.best_params, early_stopping_rounds=50)
best_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

final_metrics = {
    'train': regression_metrics(y_train, best_model.predict(X_train)),
    'val':   regression_metrics(y_val,   best_model.predict(X_val)),
    'test':  regression_metrics(y_test,  best_model.predict(X_test)),
    'best_params': study.best_params,
}
save_model(best_model, MODEL_DIR / 'xgboost.pkl')
save_json(final_metrics, RES_DIR / 'xgboost.json')
final_metrics

## 3. SHAP analysis

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(best_model)
sample = X_test.sample(min(2000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(sample)

FIG_DIR = Path('../results/figures'); FIG_DIR.mkdir(parents=True, exist_ok=True)
shap.summary_plot(shap_values, sample, show=False)
plt.tight_layout(); plt.savefig(FIG_DIR / 'xgboost_shap.png', dpi=120, bbox_inches='tight'); plt.show()